In [1]:
import json
from typing import Dict, List, Tuple
from collections import defaultdict

## 统计反先验文本替换比例

In [10]:
import re

def count_modified_words(text1, text2):
    # 使用正则表达式提取所有单词（忽略大小写）
    words1 = re.findall(r'\b\w+\b', text1.lower())
    words2 = re.findall(r'\b\w+\b', text2.lower())
    
    # if len(words1) != len(words2):
    #     print("警告：两段文本的单词总数不一致，可能影响准确性。")
    
    total_words = min(len(words1), len(words2))  # 取较小的单词数以避免索引错误
    modified_count = 0
    
    for w1, w2 in zip(words1[:total_words], words2[:total_words]):
        if w1 != w2:
            modified_count += 1
    
    if total_words == 0:
        modification_ratio = 0.0
    else:
        modification_ratio = (modified_count / total_words) * 100
    
    return modified_count, modification_ratio

In [11]:
with open("../fox_data/data.json", "r") as f:
    data = json.load(f)
    
total_modification_ratio = 0
for item in data:
    gt_text = item["gt_text"]
    distorted_text = item["distorted_text"]
    modified_count, modification_ratio = count_modified_words(gt_text, distorted_text)
    total_modification_ratio += modification_ratio
    print(f"Image: {item['image']}")
    print(f"Modified Words: {modified_count}")
    print(f"Modification Ratio: {modification_ratio:.2f}%")
    print("-" * 40)
print(f"Average Modification Ratio: {total_modification_ratio / len(data):.2f}%")

Image: en_1.png
Modified Words: 590
Modification Ratio: 70.66%
----------------------------------------
Image: en_2.png
Modified Words: 110
Modification Ratio: 10.59%
----------------------------------------
Image: en_3.png
Modified Words: 595
Modification Ratio: 84.64%
----------------------------------------
Image: en_4.png
Modified Words: 540
Modification Ratio: 86.82%
----------------------------------------
Image: en_5.png
Modified Words: 317
Modification Ratio: 59.36%
----------------------------------------
Image: en_6.png
Modified Words: 697
Modification Ratio: 96.81%
----------------------------------------
Image: en_7.png
Modified Words: 224
Modification Ratio: 41.95%
----------------------------------------
Image: en_8.png
Modified Words: 69
Modification Ratio: 9.04%
----------------------------------------
Image: en_9.png
Modified Words: 770
Modification Ratio: 99.87%
----------------------------------------
Image: en_10.png
Modified Words: 538
Modification Ratio: 99.45%
--

## 统计OCR结果

In [3]:
def analyze_evaluation_results(file_path: str) -> Tuple[Dict[str, Dict[str, float]], float]:
    """
    分析评估结果文件,按token_count分组统计precision
    
    Args:
        file_path: JSON评估结果文件路径
        
    Returns:
        分组统计结果字典(包含平均precision和样本数)和整体平均precision
    """
    # 读取JSON文件
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 用于存储每个分组的precision值
    grouped_precisions = defaultdict(list)
    
    # 遍历数据(除了最后一个overall_metrics)
    for item in data[:-1]:
        if 'token_count' in item and 'precision' in item:
            token_count = item['token_count']
            precision = item['precision']
            
            # 确定所属的分组(向下取整到百位)
            group_start = (token_count // 100) * 100
            group_end = group_start + 100
            group_key = f"{group_start}-{group_end}"
            
            grouped_precisions[group_key].append(precision)
    
    # 计算每个分组的平均precision和样本数，并按起始数值排序
    group_stats = {}
    # 按group_start数值排序
    sorted_groups = sorted(grouped_precisions.items(), key=lambda x: int(x[0].split('-')[0]))
    
    for group_key, precisions in sorted_groups:
        avg_precision = sum(precisions) / len(precisions)
        sample_count = len(precisions)
        group_stats[group_key] = {
            'avg_precision': avg_precision,
            'sample_count': sample_count
        }
    
    # 获取整体平均precision
    overall_precision = data[-1]['overall_metrics']['precision']
    
    return group_stats, overall_precision


def print_analysis_results(file_path: str):
    """
    打印分析结果
    
    Args:
        file_path: JSON评估结果文件路径
    """
    group_results, overall_precision = analyze_evaluation_results(file_path)
    
    print("=" * 70)
    print("按Token数量分组的Precision统计")
    print("=" * 70)
    
    for group_range, stats in group_results.items():
        avg_precision = stats['avg_precision']
        sample_count = stats['sample_count']
        print(f"Token范围 {group_range:12s} 样本数 = {sample_count:3d} 平均Precision = {avg_precision:.4f} ")
    
    print("=" * 70)
    print(f"整体平均Precision: {overall_precision:.4f}")
    print("=" * 70)

In [3]:
print_analysis_results("../results/ocr/en_png_tiny_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.9593 
Token范围 700-800      样本数 =  28 平均Precision = 0.9381 
Token范围 800-900      样本数 =  28 平均Precision = 0.9191 
Token范围 900-1000     样本数 =  14 平均Precision = 0.8419 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.7924 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.7434 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.5873 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.6934 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.7670 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.3441 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.5814 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.3443 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.0043 
整体平均Precision: 0.8388


In [4]:
print_analysis_results("../results/ocr/en_png_small_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.9830 
Token范围 700-800      样本数 =  28 平均Precision = 0.9698 
Token范围 800-900      样本数 =  28 平均Precision = 0.9665 
Token范围 900-1000     样本数 =  14 平均Precision = 0.9668 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.9125 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.8921 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.8644 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.9098 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.9603 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.7601 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.8618 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.7629 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.3724 
整体平均Precision: 0.9389


In [26]:
print_analysis_results("../results/ocr/en_png_raw_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.9211 
Token范围 700-800      样本数 =  28 平均Precision = 0.8551 
Token范围 800-900      样本数 =  28 平均Precision = 0.8118 
Token范围 900-1000     样本数 =  14 平均Precision = 0.8338 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.8089 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.8865 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.6328 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.6589 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.9333 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.5140 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.5803 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.9301 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.0287 
整体平均Precision: 0.8097


In [5]:
print_analysis_results("../results/ocr/from_text_tiny_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.9864 
Token范围 700-800      样本数 =  28 平均Precision = 0.9670 
Token范围 800-900      样本数 =  28 平均Precision = 0.9446 
Token范围 900-1000     样本数 =  14 平均Precision = 0.8773 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.8618 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.8072 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.7397 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.6422 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.6448 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.5359 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.6450 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.6000 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.1366 
整体平均Precision: 0.8800


In [6]:
print_analysis_results("../results/ocr/from_text_small_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.9900 
Token范围 700-800      样本数 =  28 平均Precision = 0.9848 
Token范围 800-900      样本数 =  28 平均Precision = 0.9749 
Token范围 900-1000     样本数 =  14 平均Precision = 0.9694 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.9524 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.9421 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.9185 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.8761 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.9086 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.8441 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.8701 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.7434 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.1361 
整体平均Precision: 0.9523


In [6]:
print_analysis_results("../results/ocr/from_text_raw_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.7988 
Token范围 700-800      样本数 =  28 平均Precision = 0.7007 
Token范围 800-900      样本数 =  28 平均Precision = 0.7696 
Token范围 900-1000     样本数 =  14 平均Precision = 0.7482 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.6765 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.6393 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.8292 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.8570 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.7633 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.6204 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.7533 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.8108 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.4226 
整体平均Precision: 0.7334


In [7]:
print_analysis_results("../results/ocr/distort_tiny_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.7141 
Token范围 700-800      样本数 =  28 平均Precision = 0.6254 
Token范围 800-900      样本数 =  28 平均Precision = 0.6149 
Token范围 900-1000     样本数 =  14 平均Precision = 0.5844 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.6296 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.4972 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.4470 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.4836 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.0177 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.4206 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.3451 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.2593 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.4257 
整体平均Precision: 0.5826


In [ ]:
print_analysis_results("../results/ocr/distort_small_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.7205 
Token范围 700-800      样本数 =  28 平均Precision = 0.6412 
Token范围 800-900      样本数 =  28 平均Precision = 0.6431 
Token范围 900-1000     样本数 =  14 平均Precision = 0.6835 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.7054 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.7161 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.6287 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.6651 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.4410 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.6941 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.4488 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.2000 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.5443 
整体平均Precision: 0.6551


In [7]:
print_analysis_results("../results/ocr/distort_raw_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.5865 
Token范围 700-800      样本数 =  28 平均Precision = 0.4924 
Token范围 800-900      样本数 =  28 平均Precision = 0.5447 
Token范围 900-1000     样本数 =  14 平均Precision = 0.4867 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.5675 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.5511 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.4544 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.5640 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.4805 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.3811 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.3474 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.0000 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.3478 
整体平均Precision: 0.5137


In [4]:
print_analysis_results("../results/distort/distort_tiny_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.9623 
Token范围 700-800      样本数 =  28 平均Precision = 0.8794 
Token范围 800-900      样本数 =  28 平均Precision = 0.8846 
Token范围 900-1000     样本数 =  14 平均Precision = 0.7006 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.7454 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.5776 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.5426 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.5049 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.0194 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.4675 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.4412 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.2593 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.4059 
整体平均Precision: 0.7675


In [6]:
print_analysis_results("../results/distort/distort_small_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.9878 
Token范围 700-800      样本数 =  28 平均Precision = 0.9408 
Token范围 800-900      样本数 =  28 平均Precision = 0.9691 
Token范围 900-1000     样本数 =  14 平均Precision = 0.9011 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.9205 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.8711 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.9023 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.8122 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.7333 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.8046 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.7954 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.2000 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.6392 
整体平均Precision: 0.9156


In [7]:
print_analysis_results("../results/distort/distort_raw_eval.json")

按Token数量分组的Precision统计
Token范围 600-700      样本数 =   7 平均Precision = 0.7579 
Token范围 700-800      样本数 =  28 平均Precision = 0.6704 
Token范围 800-900      样本数 =  28 平均Precision = 0.7419 
Token范围 900-1000     样本数 =  14 平均Precision = 0.6980 
Token范围 1000-1100    样本数 =  11 平均Precision = 0.7639 
Token范围 1100-1200    样本数 =   8 平均Precision = 0.7164 
Token范围 1200-1300    样本数 =   4 平均Precision = 0.6349 
Token范围 1300-1400    样本数 =   5 平均Precision = 0.8255 
Token范围 1400-1500    样本数 =   1 平均Precision = 0.8745 
Token范围 1500-1600    样本数 =   2 平均Precision = 0.5374 
Token范围 1600-1700    样本数 =   2 平均Precision = 0.6020 
Token范围 1700-1800    样本数 =   1 平均Precision = 0.0000 
Token范围 2400-2500    样本数 =   1 平均Precision = 0.4783 
整体平均Precision: 0.7058


## 统计QA结果

In [9]:
import numpy as np

In [10]:
def qa_eval(y_true, y_pred):
    pass

In [25]:
with open("../results/qa/qa_recheck_single_mode__DeepSeek-OCR.json", 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# 统计TP, FP, TN, FN，并根据此计算准确率、精确率、召回率和F1分数
TP = 0
FP = 0
TN = 0
FN = 0

for item in qa_data:
    qa_pairs = item["qa_pairs"]
    for qa in qa_pairs:
        pred_answer = qa["LLMAnswer"]
        if pred_answer is None:
            clean_answer = ""
        else:
            clean_answer = pred_answer[0].strip()
        if clean_answer == qa["correct_answer"]:
            TP += 1
        else:
            FP += 1

# 计算指标
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1_score:.4f}")

Accuracy: 0.2768
Precision: 0.2768
Recall: 1.0000
F1 Score: 0.4336


In [22]:
with open("../results/qa/qa_recheck_single_mode__Qwen2.5-3B-Instruct.json", 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# 统计TP, FP, TN, FN，并根据此计算准确率、精确率、召回率和F1分数
TP = 0
FP = 0
TN = 0
FN = 0

for item in qa_data:
    qa_pairs = item["qa_pairs"]
    for qa in qa_pairs:
        pred_answer = qa["LLMAnswer"]
        if pred_answer is None:
            clean_answer = ""
        else:
            clean_answer = pred_answer[0].strip()
        if clean_answer == qa["correct_answer"]:
            TP += 1
        else:
            FP += 1

# 计算指标
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1_score:.4f}")

Accuracy: 0.9405
Precision: 0.9405
Recall: 1.0000
F1 Score: 0.9693


In [23]:
with open("../results/qa/qa_recheck_single_mode__Qwen3-4B-Instruct-2507.json", 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# 统计TP, FP, TN, FN，并根据此计算准确率、精确率、召回率和F1分数
TP = 0
FP = 0
TN = 0
FN = 0

for item in qa_data:
    qa_pairs = item["qa_pairs"]
    for qa in qa_pairs:
        pred_answer = qa["LLMAnswer"]
        if pred_answer is None:
            clean_answer = ""
        else:
            clean_answer = pred_answer[0].strip()
        if clean_answer == qa["correct_answer"]:
            TP += 1
        else:
            FP += 1

# 计算指标
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1_score:.4f}")

Accuracy: 0.9881
Precision: 0.9881
Recall: 1.0000
F1 Score: 0.9940


In [24]:
with open("../results/qa/qa_recheck_single_mode__Llama-3.2-3B-Instruct.json", 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# 统计TP, FP, TN, FN，并根据此计算准确率、精确率、召回率和F1分数
TP = 0
FP = 0
TN = 0
FN = 0

for item in qa_data:
    qa_pairs = item["qa_pairs"]
    for qa in qa_pairs:
        pred_answer = qa["LLMAnswer"]
        if pred_answer is None:
            clean_answer = ""
        else:
            clean_answer = pred_answer[0].strip()
        if clean_answer == qa["correct_answer"]:
            TP += 1
        else:
            FP += 1

# 计算指标
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1_score:.4f}")

Accuracy: 0.9226
Precision: 0.9226
Recall: 1.0000
F1 Score: 0.9598


## 统计 VQA

In [16]:
def eval_vqa(file_path:str):
    with open(file_path, 'r', encoding='utf-8') as f:
        qa_data = json.load(f)

    count = 0

    # 统计TP, FP, TN, FN，并根据此计算准确率、精确率、召回率和F1分数
    TP = 0
    FP = 0
    TN = 0
    FN = 0

    for item in qa_data:
        qa_pairs = item["qa_pairs"]
        for qa in qa_pairs:
            pred_answer = qa["LLMAnswer"]
            if pred_answer is None:
                clean_answer = ""
            else:
                clean_answer = pred_answer[0].strip()
                if clean_answer not in ["A", "B", "C", "D"]:
                    clean_answer = ""
                    count += 1
                    # break
            if clean_answer == qa["correct_answer"]:
                TP += 1
            else:
                FP += 1

    # 计算指标
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Unexpected answers count: {count}")

In [17]:
eval_vqa("../results/vqa/en_png_tiny.json")

Accuracy: 0.1994
Unexpected answers count: 33


In [18]:
eval_vqa("../results/vqa/en_png_small.json")

Accuracy: 0.1845
Unexpected answers count: 9


In [19]:
eval_vqa("../results/vqa/en_png_raw.json")

Accuracy: 0.2143
Unexpected answers count: 0


In [20]:
eval_vqa("../results/vqa/from_text_tiny.json")

Accuracy: 0.1964
Unexpected answers count: 9


In [21]:
eval_vqa("../results/vqa/from_text_small.json")

Accuracy: 0.2143
Unexpected answers count: 0


In [23]:
eval_vqa("../results/vqa/from_text_raw.json")

Accuracy: 0.2232
Unexpected answers count: 1


In [24]:
eval_vqa("../results/vqa/distort_tiny.json")

Accuracy: 0.1994
Unexpected answers count: 6


In [25]:
eval_vqa("../results/vqa/distort_small.json")

Accuracy: 0.2232
Unexpected answers count: 2


In [26]:
eval_vqa("../results/vqa/distort_raw.json")

Accuracy: 0.2173
Unexpected answers count: 1


## 统计replace

In [27]:
with open('../fox_data/replace.json', 'r') as f:
    data = json.load(f)
for item in data:
    print(item["replace"])

 
FREEDOM OF INFORMATION ACT (EXCERPT) 
Act 442 of 1976 
15.240.amended Options by requesting person; appeal; actions by public **; receipt of written appeal; judicial review; civil 
action; venue; de ** proceeding; burden of proof; private view of public record; contempt; assignment of action or appeal 
for hearing, trial, or argument; attorneys' fees, costs, and disbursements; assessment of award; damages. 
Sec. 10. 
(1) If a public body makes a final determination to deny all or a portion of a request, ** requesting person may do 1 of the following at 
his or her option: 
(a) Submit to the head of the public body a written appeal that specifically states ** word "appeal" and identifies the reason or 
reasons for reversal of the denial. 
 
(b) Commence a ** ** in the circuit court, or if ** decision of a state public body ** at issue, the court of claims, to 
compel the public body's disclosure of ** public records within 180 days after a public body's final determination to deny a 
